# 06 — Media Coverage Trends & Peak Detection

Produces:
- A bar chart of the top media outlets covering the league.
- A monthly coverage trend chart (with a linear trendline).
- A statistical peak-detection report (`scipy.find_peaks`) for both peak **months** and peak **days**, with sample headlines, exported to CSV.

> **Note on a fix made here:** the original `mediacloud_clean.py` printed a peak-months report but never saved it to CSV, and saved the peak-days report under a filename (`daily_article_peaks.csv`) that didn't match what the topic-modeling script expected to read (`daily_peaks.csv`, `monthly_peaks.csv`, both including a `prominence` column). This notebook exports both files, under the names and with the columns `07_media_topic_modeling.ipynb` actually needs, so the pipeline runs end-to-end.

Uses the shared color theme in `plot_theme.py` so every chart in this repo reads as part of the same project.

**Inputs:** `data/processed/nwsl_articles_filtered.csv`
**Outputs:** `output/nwsl_articles_by_media_outlet.png`, `output/nwsl_articles_trend_over_time.png`, `data/processed/monthly_peaks.csv`, `data/processed/daily_peaks.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import find_peaks
import os

from plot_theme import set_mpl_theme

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUT_DIR = os.path.join("..", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

set_mpl_theme()

media = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "nwsl_articles_filtered.csv"))
media["publish_date"] = pd.to_datetime(media["publish_date"])
print(f"Loaded {len(media)} filtered articles")


## Top media outlets covering the NWSL

In [ ]:
def plot_top_outlets(media, top_n=20, out_path=None):
    """Bar chart of the top N media outlets by article count."""
    plt.figure(figsize=(12, 6))
    media["media_name"].value_counts().head(top_n).plot(kind="bar")
    plt.title(f"Top {top_n} Media Outlets Covering NWSL")
    plt.xlabel("Media Outlet")
    plt.ylabel("Number of Articles")
    plt.xticks(rotation=45)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path)
    plt.show()
    plt.close()


plot_top_outlets(media, out_path=os.path.join(OUTPUT_DIR, "nwsl_articles_by_media_outlet.png"))


## Monthly coverage trend

In [ ]:
def monthly_counts(media):
    """Return a Series of article counts indexed by month (Timestamp)."""
    counts = media["publish_date"].dt.to_period("M").value_counts().sort_index()
    counts.index = counts.index.to_timestamp()
    return counts


def plot_monthly_trend(monthly, out_path=None):
    """Line chart of monthly article counts with a linear trendline."""
    x = monthly.index
    y = monthly.values
    x_numeric = np.arange(len(x))

    slope, intercept = np.polyfit(x_numeric, y, 1)
    y_trend = slope * x_numeric + intercept

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(x, y, label="Article Count")
    ax.plot(x, y_trend, linewidth=2, color="#e34948", linestyle="--", alpha=0.8, label="Trend")

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 7)))

    ax.set_title("NWSL Media Coverage Trend Since League Announcement")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of Articles per Month")
    plt.xticks(rotation=45, ha="right")
    ax.legend(loc="upper left", frameon=False, fontsize=11, labelcolor="#52514e")
    ax.set_facecolor("white")

    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.show()
    plt.close(fig)


monthly = monthly_counts(media)
plot_monthly_trend(monthly, out_path=os.path.join(OUTPUT_DIR, "nwsl_articles_trend_over_time.png"))


## Peak detection (months and days)

In [ ]:
def find_significant_peaks(x, y, prominence_factor, distance):
    """Run scipy.find_peaks and return results as a sorted DataFrame."""
    peaks, properties = find_peaks(y, prominence=np.std(y) * prominence_factor, distance=distance)
    peak_df = pd.DataFrame({
        "date": x[peaks],
        "article_count": y[peaks],
        "prominence": properties["prominences"],
    }).sort_values("article_count", ascending=False)
    return peak_df


In [ ]:
def report_monthly_peaks(media, monthly, top_n=10, out_path=None):
    """Print a report of statistically significant peak months with sample headlines,
    and export the peaks (with a day_of_week-equivalent month label) to CSV."""
    x = monthly.index
    y = monthly.values

    peak_df = find_significant_peaks(x, y, prominence_factor=0.5, distance=3)
    threshold = np.mean(y) + 1.5 * np.std(y)

    print("=== SIGNIFICANT PEAKS IN ARTICLE FREQUENCY ===\n")
    print(f"Average monthly articles: {np.mean(y):.0f}")
    print(f"Standard deviation: {np.std(y):.0f}")
    print(f"Threshold for significance: {threshold:.0f}\n")
    print(peak_df.to_string(index=False))
    print(f"\nTotal significant peaks found: {len(peak_df)}")

    print("\n=== PEAK DETAILS ===")
    for _, row in peak_df.head(top_n).iterrows():
        month_str = row["date"].strftime("%B %Y")
        print(f"\n{month_str}: {row['article_count']} articles")

        month_period = row["date"].to_period("M")
        month_mask = media["publish_date"].dt.to_period("M") == month_period
        sample_titles = media.loc[month_mask, "title"].head(3)
        for title in sample_titles:
            print(f"  \u2022 {title[:80]}...")

    if out_path:
        peak_df.to_csv(out_path, index=False)
        print(f"\nExported {len(peak_df)} peak months to '{out_path}'")

    return peak_df


monthly_peaks = report_monthly_peaks(
    media, monthly, out_path=os.path.join(DATA_PROCESSED_DIR, "monthly_peaks.csv")
)


In [ ]:
def report_daily_peaks(media, top_n=15, out_path=None):
    """Print a report of significant peak days with headlines, and export to CSV."""
    daily = media["publish_date"].dt.date.value_counts().sort_index()
    x_daily = daily.index.to_numpy()
    y_daily = daily.values

    peak_df = find_significant_peaks(x_daily, y_daily, prominence_factor=0.8, distance=2)

    print("=== SIGNIFICANT PEAK DAYS ===\n")
    print(f"Average daily articles: {np.mean(y_daily):.1f}")
    print(f"Max articles in a single day: {np.max(y_daily)}")
    print(f"Standard deviation: {np.std(y_daily):.1f}\n")

    print(f"TOP {top_n} PEAK DAYS:\n")
    for rank, (_, row) in enumerate(peak_df.head(top_n).iterrows(), 1):
        date_str = pd.to_datetime(row["date"]).strftime("%A, %B %d, %Y")
        count = int(row["article_count"])
        print(f"{rank}. {date_str} - {count} articles")

        day_articles = media[media["publish_date"].dt.date == row["date"]]
        print(f"   Sources: {day_articles['media_name'].unique()[:3].tolist()}")
        for _, article in day_articles.head(4).iterrows():
            print(f"   \u2022 {article['title'][:70]}...")
        print()

    # Keep "date", "article_count", and "prominence" -- topic modeling
    # notebook needs prominence to rank peaks -- plus day_of_week for readability
    export_df = peak_df[["date", "article_count", "prominence"]].copy()
    export_df["day_of_week"] = pd.to_datetime(export_df["date"]).dt.day_name()

    if out_path:
        export_df.to_csv(out_path, index=False)
        print(f"\nExported {len(export_df)} peak days to '{out_path}'")

    return peak_df


daily_peaks = report_daily_peaks(
    media, out_path=os.path.join(DATA_PROCESSED_DIR, "daily_peaks.csv")
)
